# Pilot XAI-DSS trên IndPenSim — bản v2

So với bản v1, bản này có:
- dải hành động rộng hơn (×0,6 – ×1,6);
- ràng buộc thể tích bình 100.000 L;
- hai quy tắc cố định để so sánh với DSS;
- quy mô mặc định 120 mẻ huấn luyện và 24 mẻ kiểm thử.

Mọi tính toán chạy **trên máy chủ của Google**, máy tính của bạn chỉ hiển thị trang web.

**Trước khi chạy:**
1. Upload `pilot_colab_v2.zip` và notebook này vào cùng thư mục với bản cũ trên Google Drive, ví dụ `MyDrive/De_Tai_Nong_San/`.
2. Chọn *Runtime → Change runtime type → CPU*. Không cần GPU.
3. Chạy lần lượt từng ô từ trên xuống.

**Nếu bị ngắt kết nối:** chạy lại các ô 1–3, rồi chạy lại ô đang dở. Những mẻ đã xong được lưu trên Drive và sẽ được bỏ qua.

## Ô 1. Kết nối Google Drive và giải nén code vào `pilot_v2/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile
DRIVE_DIR = '/content/drive/MyDrive/De_Tai_Nong_San'   # sửa nếu bạn để file ở thư mục khác
ZIP_PATH = os.path.join(DRIVE_DIR, 'pilot_colab_v2.zip')
WORK = os.path.join(DRIVE_DIR, 'pilot_v2')

if not os.path.exists(os.path.join(WORK, 'sim.py')):
    zipfile.ZipFile(ZIP_PATH).extractall(DRIVE_DIR)
    print('Đã giải nén vào', WORK)
os.chdir(WORK)
os.environ['PILOT_EXP'] = 'v2'      # kết quả ghi vào runs/v2/
print('Thư mục làm việc:', os.getcwd())
print(sorted(os.listdir('.')))

## Ô 2. Cài thư viện (thường Colab đã có sẵn)

In [ ]:
!pip install -q lightgbm shap
import numpy, scipy, pandas, sklearn, lightgbm, shap
print('numpy', numpy.__version__, '| scipy', scipy.__version__, '| lightgbm', lightgbm.__version__, '| shap', shap.__version__)

## Ô 3. Chọn quy mô chạy

| Chế độ | Mẻ huấn luyện | Mẻ kiểm thử | Thời gian (2 CPU) |
|---|---|---|---|
| `NHANH` | 50 | 10 | khoảng 1–1,5 giờ |
| `DAY_DU` (mặc định) | 120 | 24 | khoảng 2–3 giờ |

In [ ]:
CHE_DO = 'DAY_DU'         # 'DAY_DU' hoặc 'NHANH'
N_TRAIN, N_TEST = (120, 24) if CHE_DO == 'DAY_DU' else (50, 10)
print('Chế độ:', CHE_DO, '| mẻ huấn luyện:', N_TRAIN, '| mẻ kiểm thử:', N_TEST, '| số CPU:', os.cpu_count())

## Ô 4. Kiểm tra nhanh bộ mô phỏng (khoảng 1 phút)

Kết quả phải giống bản v1: seed 1 ≈ **3640 kg**, seed 2 ≈ **3258 kg**, seed 3 ≈ **2705 kg**.

In [ ]:
!python -W ignore sim.py

## Ô 5. Sinh dữ liệu huấn luyện (phần lâu nhất, khoảng 1 giờ ở chế độ `DAY_DU`)

In [ ]:
!python -W ignore gen_data.py --seeds {N_TRAIN}

## Ô 6. Huấn luyện hai mô hình dự báo và chọn hành động cố định tốt nhất (vài phút)

Kiểm định chéo chia theo mẻ: không mẻ nào vừa nằm trong tập huấn luyện vừa nằm trong tập kiểm định.

In [ ]:
!python train_dss.py

## Ô 7. Chạy 4 chính sách trên các mẻ kiểm thử và kiểm chứng lời giải thích (khoảng 1 giờ)

Mỗi mẻ kiểm thử chạy theo: công thức, quy tắc "đường và dầu ×1,2", hành động cố định tốt nhất, và DSS.

In [ ]:
!python -W ignore eval_dss.py --seeds {N_TEST}

## Ô 8. Tổng hợp số liệu

In [ ]:
!python analyze.py

## Ô 9. Gom kết quả thành `ket_qua_pilot_v2.zip` trên Drive

Tải file này về, đặt vào `D:\De_Tai_Nong_San\`, rồi báo Claude để phân tích và viết báo cáo cho giáo sư.

In [ ]:
out = os.path.join(DRIVE_DIR, 'ket_qua_pilot_v2.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['runs/v2/results', 'runs/v2/models']:
        for root, _, files in os.walk(folder):
            if '/parts' in root:
                continue
            for f in files:
                z.write(os.path.join(root, f))
    if os.path.exists('runs/v2/data/train.csv'):
        z.write('runs/v2/data/train.csv')
print('Đã lưu', out, f'({os.path.getsize(out)/1e6:.1f} MB)')